# Jour 1 : Architecturer un système IA moderne — Du Prototype au Service Production

## 1. Introduction

Jusqu'à la fin de la quatrième semaine, nos développements se sont concentrés sur la logique décisionnelle interne de nos agents. Nous avons écrit un micro-framework, encapsulé des fonctions Python par introspection et structuré des graphes d'états. C'est une phase d'ingénierie applicative indispensable, mais elle s'exécutait au sein d'un environnement "fermé" (CLI local ou notebooks).

La transition d'un prototype d'agent exécuté en ligne de commande vers un service de production capable d'accueillir des milliers d'utilisateurs concurrents est l'un des défis les plus complexes de l'ingénierie d'IA moderne. En production, un agent ne peut pas être une simple classe Python persistante en mémoire serveur. Il doit faire face à des déconnexions réseau, à des timeouts d'API LLM, à la gestion asynchrone du streaming de texte, et à la persistance sécurisée des sessions de discussion.

Aujourd'hui, nous posons les fondations théoriques et architecturales de notre infrastructure de production. Nous allons analyser précisément pourquoi les backends d'agents violent les règles classiques du REST, comment modéliser l'état conversationnel, et concevoir notre architecture système cible impliquant FastAPI, Redis, PostgreSQL et le protocole MCP.

## 2. Où en sommes-nous dans le cursus?

- **Semaine 4 (Frameworks d'Agents)** : Nous avons conçu notre micro-framework mini_framework et étudié les orchestrateurs du marché (LangGraph, OpenAI SDK, CrewAI).

- **Semaine 5 — Jour 1 (Aujourd'hui - Ce module)**: Nous dessinons l'architecture système distribuée et analysons la gestion d'état (Stateful vs Stateless).

- **Semaine 5 — Jour 2 (Demain)** : Nous implémenterons l'API FastAPI asynchrone capable de diffuser les réponses de l'agent en temps réel via des flux Server-Sent Events (SSE).

- **Semaine 5 — Jours 3 à 7** : Nous aborderons la dockerisation, la mise à l'échelle via Kubernetes, l'observabilité OpenTelemetry et la sécurité offensive/défensive de nos agents.

## 3. Objectifs pédagogiques
À l'issue de cette journée, vous serez capable de :

 - **Démontrer en quoi une architecture applicative d'IA** diffère d'un backend d'API REST traditionnel.

- **Modéliser la gestion d'état** (Stateful) d'un agent au-dessus d'une couche API sans état (Stateless).

- **Concevoir un schéma de données robuste** pour stocker l'historique conversationnel (Redis pour le cache chaud, PostgreSQL pour le stockage froid).

- **Justifier le cycle de vie d'une requête d'agent** à travers une pile technologique distribuée.

- **Réaliser l'audit d'une architecture cible** et concevoir des diagrammes de flux d'exécution industriels.

## 4. Pourquoi une architecture IA est-elle différente d'un backend REST classique?

Le paradigme REST (Representational State Transfer) a été conçu pour l'échange de ressources légères et sans état (stateless) via des requêtes HTTP courtes. Cette approche présente d'importantes limites lorsqu'elle est confrontée aux contraintes des agents d'intelligence artificielle :

|Caractéristique|Backend REST Classique|Architecture Applicative d'IA (Agent)|
|---|---|---|
|Durée d'une requête|Ultra-rapide (millisecondes).|Très longue (secondes à minutes, le temps que le LLM raisonne et appelle ses outils).|
|Gestion de l'état|Entièrement Stateless. Chaque requête HTTP contient l'intégralité des informations nécessaires à sa résolution.|Fortement Stateful. L'agent doit se souvenir de l'historique pour maintenir la cohérence de sa boucle ReAct.|
|Consommation de ressources|Prévisible (liaisons de base de données légères, opérations d'E/S simples).|Imprévisible (pics de latence réseau, variations de taille de contexte, calculs vectoriels coûteux).|
|Flux de sortie (Output)|Réponse JSON complète envoyée d'un seul bloc à la fin du traitement.|Flux en temps réel (Streaming) nécessaire pour éviter l'effet "écran gelé" pour l'utilisateur.|
|Coût d'exécution|Quasi-nul par requête d'E/S.|Indexé sur la fenêtre de contexte et la consommation de jetons (Tokens).|


## 5. Les Composants d'un Système Agentique de Production

Pour répondre à ces contraintes, nous devons structurer notre système en plusieurs couches logiques découplées, assurant chacune une responsabilité unique :
```
                                  Couches d'Architecture
   +---------------------------------------------------------------------------------+
   |  [Couche Client] : Interfaces Web, applications mobiles, terminaux clients.    |
   |                                          |                                      |
   |  [Couche Gateway] : API Gateway (routage, sécurité, limites de requêtes).      |
   |                                          |                                      |
   |  [Couche API Application] : FastAPI (asynchronisme, orchestration, streaming).  |
   |                                          |                                      |
   |  [Couche d'État] : Redis (sessions chaudes) & PostgreSQL (historique froid).    |
   |                                          |                                      |
   |  [Couche Intégration] : Serveurs d'outils MCP, exécuteurs de code sandboxés.    |
   |                                          |                                      |
   |  [Couche d'Inférence] : Fournisseurs de modèles de langage (LLM Cloud/Local).   |
   +---------------------------------------------------------------------------------+
```

## 6. Stateless vs Stateful : Le Grand Arbitrage

Les modèles de langage (LLM) sont, par conception, des fonctions mathématiques strictement sans état (Stateless). Ils prennent une séquence de jetons en entrée et prédisent la probabilité du jeton suivant en sortie, sans conserver la moindre mémoire des requêtes passées.

Pour donner l'illusion d'une conversation cohérente, notre système applicatif doit être étatique (Stateful). La modélisation de cette transition d'état s'appuie sur des choix de stockage distincts :

- **L'état éphémère (Mémoire de Calcul - RAM)** : Stocker l'historique d'une conversation directement dans la mémoire vive de votre serveur web (par exemple, dans une variable globale ou un dictionnaire Python) est un anti-pattern absolu de production. Si votre serveur effectue un redémarrage, ou si vous lancez plusieurs instances de votre API derrière un répartiteur de charge (Load Balancer), l'utilisateur perdra instantanément sa session.

- **Le stockage de session chaud (Redis)** : Utilisé pour stocker l'historique conversationnel actif et les verrous de synchronisation. Redis offre des temps de lecture/écriture inférieurs à la milliseconde, parfaits pour recharger la session à chaque tour de parole.

- **Le stockage à long terme (PostgreSQL)** : Utilisé pour archiver de manière durable les sessions clôturées, stocker les donné

## 7. Gestion des Sessions : Où vivent les conversations?

Une conversation en production doit être représentée par une structure de données d'état stricte.

**Modélisation Mathématique de la Reconstitution d'État**

Soit $S$ l'espace des états conversationnels d'un système. Puisque notre API web FastAPI est stateless (sans état physique persistant en mémoire vive d'instance), elle doit reconstituer l'historique de discussion à chaque nouvelle requête HTTP de l'utilisateur.

Soit un identifiant de session unique $\text{sid}$. Nous définissons l'opérateur de reconstitution d'état $R$ qui interroge notre base de données Redis $\mathcal{D}_{\text{Redis}}$ :
$$H_k = R(\text{sid}, \mathcal{D}_{\text{Redis}})$$

Où $H_k = \langle M_1, M_2, \dots, M_k \rangle$ représente la séquence ordonnée de messages représentant la session active de l'utilisateur.

Lorsque l'utilisateur envoie une nouvelle consigne $U_{k+1}$, l'API FastAPI exécute la transition d'état suivante :
1. Ajout de la requête utilisateur à l'historique :$$H'_k = H_k \cup \{ M_{\text{user}}(U_{k+1}) \}$$
2. Appel de la politique décisionnelle du LLM $\pi_\theta$ :$$M_{\text{assistant}} \sim \pi_\theta(H'_k)$$
3. Mise à jour et sérialisation asynchrone de l'état dans Redis :$$H_{k+1} = H'_k \cup \{ M_{\text{assistant}} \}$$$$\mathcal{D}'_{\text{Redis}} = \text{Sauvegarder}(\text{sid}, H_{k+1}, \mathcal{D}_{\text{Redis}})$$

## 8. Gestion des Instances : Où vivent les agents?

Une erreur de conception récurrente consiste à instancier un objet Agent par utilisateur et à tenter de maintenir cette instance Python persistante en mémoire sur le serveur web.

**Règle d'or de l'ingénierie des API d'agents**

"Les instances d'agents Python doivent être éphémères et sans état. Les données d'état de session doivent être persistantes et centralisées." 

1. Lorsqu'un utilisateur envoie une requête :FastAPI reçoit la requête contenant le session_id et le message de l'utilisateur.
2. FastAPI interroge Redis pour récupérer l'historique conversationnel lié à ce session_id.
3. FastAPI instancie un objet Agent à la volée, lui injecte l'historique conversationnel rechargé et le registre d'outils, puis lance l'exécution.
4. Dès que l'agent produit sa réponse, l'historique est sauvegardé dans Redis, et l'instance de l'agent est immédiatement détruite par le garbage collector de Python, libérant les ressources du serveur.Cette approche permet à votre API de passer à l'échelle horizontalement (scale-out) sans aucune contrainte d'affinité de session au niveau de votre répartiteur de charge.

## 9. Étude d'Architecture Cible : Le Diagramme de Flux

Nous allons étudier pas-à-pas le cycle de vie d'une requête au sein de notre architecture cible de production :
```
Utilisateur -> API Gateway -> FastAPI -> Agent -> (MCP, Redis, PostgreSQL) -> LLM
```
```

  Utilisateur        Gateway          FastAPI           Redis         Tool Server (MCP)      LLM
       |                |                |                |                  |                |
       |-- POST request>|                |                |                  |                |
       |   (msg, sid)   |-- Route req. ->|                |                  |                |
       |                |                |-- Fetch hist.->|                  |                |
       |                |                |<-- Return hist.|                  |                |
       |                |                |                                   |                |
       |                |                |-- [Instancier Agent éphémère]     |                |
       |                |                |-------------------- Inférence -------------------->|
       |                |                |<----------------- Demande outil -------------------|
       |                |                |                                   |                |
       |                |                |-- Exécuter outil (JSON-RPC) ----->|                |
       |                |                |<-- Retourner observation (JSON) --|                |
       |                |                |                                                    |
       |                |                |-------------------- Inférence -------------------->|
       |                |                |<----------------- Synthèse finale -----------------|
       |                |                |                                   |                |
       |                |                |-- Save State ->|                  |                |
       |<-- Response ---|<-- Streaming --|                |                  |                |
```
**Le Cycle de Vie d'une Requête :**

1. **Couche Client & Gateway** : L'utilisateur envoie son message et son identifiant de session. L'API Gateway intercepte la requête, valide l'authentification de l'utilisateur, vérifie ses quotas d'utilisation (Rate Limiting) et transmet la requête propre à FastAPI.

2. **Couche FastAPI (Reconstitution)** : FastAPI intercepte la requête, extrait l'identifiant de session et interroge la base Redis pour reconstituer l'historique.

3. **Couche Agent & Inférence** : L'agent est instancié avec l'historique d'apprentissage. Il formule sa requête d'inférence initiale vers le fournisseur LLM.

4. **Couche Intégration (MCP)** : Si le LLM formule une demande d'outil, l'agent l'intercepte et transmet la commande en format JSON-RPC sécurisé au serveur d'outils MCP concerné. Le résultat (l'observation) est réinjecté dans l'historique.

5. **Couche de Persistance & Streaming** : Une fois la boucle de décision finalisée, l'historique mis à jour est persisté de manière asynchrone dans Redis (et archivé dans PostgreSQL). FastAPI diffuse la réponse finale au client en utilisant un flux continu pour une réactivité maximale.

## 10. Schématisation d'Architecture (Détail de l'Infrastructure cible)

Voici la représentation visuelle détaillée des couches d'infrastructure que nous déploierons au cours des prochains jours :

```
                              ARCHITECTURE TECHNIQUE CIBLE
                              
                     +---------------------------------------+
                     |          Utilisateur (Web/App)        |
                     +---------------------------------------+
                                         |
                                         | HTTPS (JSON / SSE)
                                         v
                     +---------------------------------------+
                     |               API Gateway             |
                     |     (Kong / Nginx / Traefik Gateway)  |
                     +---------------------------------------+
                                         |
                                         | Routage interne
                                         v
                     +---------------------------------------+
                     |           FastAPI Web Server          |
                     |         (Workers Uvicorn / Gunicorn)  |
                     +---------------------------------------+
                       /                 |                 \
                      /                  |                  \
       Lecture/Écriture   Lecture/Écriture   Requêtes Outils   Appels Inférence
                    /                    |                    \                    \
                   v                     v                     v                    v
      +-----------------+      +-----------------+      +-----------------+      +-------------+
      |      Redis      |      |   PostgreSQL    |      |  MCP Servers    |      | LLM Provider|
      |  (Session Cache)|      | (Audit/Cold Mem)|      | (APIs, Database)|      | (OpenAI API)|
      +-----------------+      +-----------------+      +-----------------+      +-------------+
```
## 11. Le Rôle Crucial de Redis et PostgreSQL

Pour garantir une haute performance et une auditabilité complète, notre architecture sépare clairement les rôles de nos deux moteurs de bases de données :

### 11.1. Redis : La mémoire vive partagée

- **Stockage de session chaud** : L'historique des conversations actives est stocké sous forme de clés-valeurs avec un temps d'expiration automatique (TTL - Time to Live) de 2 heures. Si un utilisateur n'interagit plus pendant 2 heures, la session chaude expire, libérant l'espace mémoire de Redis.

- **Verrous distribués (Locks)** : Empêche un utilisateur d'envoyer simultanément deux requêtes sur la même session de discussion, ce qui corromprait l'ordre des messages et provoquerait des conflits logiques d'appels d'outils.

### 11.2. PostgreSQL : L'archive durable et analytique

- **Auditabilité complète** : En production, chaque message, chaque temps de réponse de LLM, chaque outil appelé et chaque volume de jetons consommés doivent être historisés à des fins réglementaires, d'audit de sécurité et d'optimisation financière.

- **Stockage de documents et vecteurs (pgvector)** : PostgreSQL servira de base de données vectorielle intégrée pour stocker les index sémantiques de notre RAG au sein d'une unique base de données d'entreprise, limitant la complexité de notre pile technologique.

## 12. 🧠 Notes d'Architecte : La gestion de la "Session Rehydration"

La réhydratation de session (Session Rehydration) consiste à transformer une structure de données sérialisée issue d'une base de données en une instance d'objet Python prête à s'exécuter au sein de notre runtime d'agent [1, 1].

**Comment implémenter une réhydratation robuste?**

Pour éviter tout couplage technique, notre API FastAPI ne doit jamais manipuler directement les objets de bases de données. Nous mettons en œuvre le pattern de **Repository** :

1. Une classe *SessionRepository* interroge la couche de stockage (Redis/Postgres).
2. Elle extrait le dictionnaire sérialisé.
3. Elle utilise un validateur de données (comme un modèle Pydantic) pour s'assurer que les données n'ont pas été altérées.
4. Elle instancie et retourne l'objet *Conversation* propre à notre framework.

Ce découplage garantit que si vous décidez demain de basculer de Redis vers MongoDB ou Cassandra, vous n'aurez qu'à réécrire la classe SessionRepository, sans jamais altérer une seule ligne de code de vos agents ou de vos endpoints FastAPI.

## 13. 📈 Notes Marché : L'évolution de la stack de persistance d'agents

En 2026, l'industrie a tiré les leçons des premières vagues d'architectures d'IA expérimentales. L'intégration de bases de données vectorielles dédiées (noSQL) décline au profit de solutions unifiées relationnelles robustes  :    
```
Tendance de Persistance d'IA (2026)
+----------------------------------------------------------------+
|  PostgreSQL (pgvector) : ⭐⭐⭐⭐⭐ (Le standard d'entreprise) |
|  Redis (Session Cache) : ⭐⭐⭐⭐⭐ (Le cache de référence)    |
|  Pinecone / Vector-only DB : ⭐⭐☆☆☆ (En perte de vitesse)     |
+----------------------------------------------------------------+
```
Les entreprises privilégient des architectures de données rationalisées. PostgreSQL s'est imposé comme l'épine dorsale des applications d'IA d'entreprise en gérant au même endroit les données relationnelles classiques, les documents JSON non structurés et les embeddings vectoriels sémantiques de haute dimension via l'extension *pgvector*.

## 14. Atelier : Dessiner votre architecture cible

**Énoncé de l'atelier**

Vous êtes nommé Architecte IA Solution pour un grand groupe bancaire. Vous devez déployer un agent autonome d'aide à la conformité réglementaire. Cet agent doit analyser des contrats de prêts immobiliers, interroger des APIs internes sécurisées de crédit, et enregistrer ses rapports d'audit.

**Consignes de modélisation**

En utilisant un outil de dessin de diagrammes (comme Excalidraw, Draw.io ou Mermaid), réalisez la schématisation complète de votre infrastructure cible :
1. Tracez les frontières de sécurité réseau (VPC, API Gateway, serveurs d'outils MCP isolés).
2. Modélisez le cycle de vie complet d'une requête de l'utilisateur de l'authentification à l'archivage en base.
3. Intégrez les composants de persistance d'état (Redis/PostgreSQL) en spécifiant le rôle de chacun.
4. Rédigez un document d'une page (Markdown) détaillant votre stratégie pour éviter les corruptions de sessions lors d'appels concurrents.

## 15. Exercice : Modélisation SQL et Redis de persistance de session
**Objectif**

Modéliser physiquement les structures de tables SQL de PostgreSQL et les clés d'enregistrement Redis nécessaires pour stocker les messages et métadonnées d'une session de conversation agentique.

**Consignes**

1. Rédigez le script d'initialisation SQL standard de votre base de données d'audit PostgreSQL.

2. Modélisez la structure d'enregistrement clé-valeur JSON de votre cache Redis.

**Script SQL de Production (PostgreSQL)**
```
SQL
-- Initialisation de la base de données d'audit d'agents IA
CREATE TABLE IF NOT EXISTS users (
    user_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    email VARCHAR(255) UNIQUE NOT NULL,
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS agent_sessions (
    session_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    user_id UUID REFERENCES users(user_id) ON DELETE CASCADE,
    agent_name VARCHAR(100) NOT NULL,
    status VARCHAR(50) DEFAULT 'active', -- active, archived, closed
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS session_messages (
    message_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    session_id UUID REFERENCES agent_sessions(session_id) ON DELETE CASCADE,
    role VARCHAR(50) NOT NULL, -- system, user, assistant, tool
    content TEXT,
    tool_calls JSONB, -- Contient les demandes d'outils au format structuré
    name VARCHAR(100), -- Nom de l'outil si le rôle est 'tool'
    tool_call_id VARCHAR(100),
    tokens_consumed INT DEFAULT 0,
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

-- Index pour accélérer la reconstitution d'historique lors d'appels d'API
CREATE INDEX idx_messages_session_id ON session_messages(session_id);
```

***Modélisation du Document de Session Redis (JSON Cache)***

Clé d'enregistrement : session:session_recherche_mcp_123

Temps d'expiration (TTL) : 7200 (2 heures)
```
JSON
{
  "session_id": "session_recherche_mcp_123",
  "user_id": "8f3b92cb-2f34-4a41-86fc-b3b42918bb1d",
  "metadata": {
    "model": "gpt-4o-mini",
    "temperature": 0.0,
    "last_interaction": "2026-07-06T16:07:00Z"
  },
  "messages":
    }
  ]
}
```

## 16. Synthèse

- **Une architecture IA viole les règles du REST classique** : Le temps d'exécution long, la cyclicité, le besoin de streaming en temps réel et la dépendance envers l'état conversationnel requièrent une architecture multi-tier asynchrone.

- **La séparation Stateful / Stateless est fondamentale** : Les instances d'agents et d'API doivent être éphémères (Stateless) pour permettre la montée en charge, tandis que l'état de la session (Stateful) est centralisé dans des bases dédiées (Redis/PostgreSQL).

- **Redis et PostgreSQL sont complémentaires** : Redis gère la performance de la session active en cache chaud, PostgreSQL assure la traçabilité réglementaire, l'analytique et la mémoire froide à long terme.

## 17. Bibliographie

**AWS Architecture Blog** — Best practices for building stateful workflows over stateless APIs

**FastAPI Official Documentation** — Concurrency and async/await design patterns : (https://fastapi.tiangolo.com/async/)

**Redis Enterprise Whitepapers** — Using Redis for Session Management in Distributed Systems

## 18. Ce qui sera développé demain

Demain (Semaine 5 — Jour 2), nous passons au code de production! Nous allons concevoir l'intégralité d'une API d'agents asynchrone en utilisant **FastAPI**.

Nous écrirons les endpoints d'exécutions et mettrons en place un pipeline de diffusion en temps réel utilisant le protocole **Server-Sent Events (SSE)**, assurant que les jetons produits par l'agent soient affichés instantanément sur le terminal de l'utilisateur.